# B2-024-gpu-scientific-ml-capstone — Practice p16 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** gpu-colab-l4-workflow

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import os
import tempfile

CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "batch_size": 64, "steps": 300, "checkpoint_at": 150,
              "lr": 0.01, "channels": (8, 16), "simulated_rows": 0}

## Solution

**Plan.** Re-implement the p06 helpers (`get_device`, `move_batch`, `train_step`) and the p07 checkpoint pair, then drive everything through one `train(config, ...)` function that reads the device, precision, sizes, and data pool from the config dict, so the L4 run changes only the dictionary.
The uninterrupted run saves a checkpoint at step 150 as a side effect; the resumed run loads it into a brand-new model and optimizer after the global generator has been moved elsewhere, and continues from step 151.
Because the checkpoint restores weights, Adam state, *and* the global RNG state, step 151 of the resumed run draws the same mini-batch as the uninterrupted run, so on the CPU every later loss is bit-identical.

**Seam.** A forward pre-hook on each model hashes every input row while `fitting` is true (only inside `train`), so the set of splits seen during fitting is recorded. Validation and test forward passes happen with `fitting = False`.

In [ ]:
from contextlib import nullcontext


# ---- p06 helpers
def get_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


def move_batch(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device)
    if isinstance(batch, (tuple, list)):
        return type(batch)(move_batch(item, device) for item in batch)
    if isinstance(batch, dict):
        return {key: move_batch(value, device) for key, value in batch.items()}
    return batch


def train_step(model, batch, optimizer, device, amp_dtype):
    images, labels = move_batch(batch, device)
    optimizer.zero_grad(set_to_none=True)
    context = nullcontext() if amp_dtype is None else torch.autocast(device_type=device.type, dtype=amp_dtype)
    with context:
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
    loss.backward()
    optimizer.step()
    return {"loss": loss.item(), "logits_dtype": logits.dtype}


# ---- p07 checkpoints
def save_checkpoint(path, model, optimizer, step):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": int(step),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, path)


def load_checkpoint(path, model, optimizer):
    state = torch.load(path, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    torch.set_rng_state(state["torch_rng"])
    if state["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    return int(state["step"])


# ---- model
class ShapeCNN(nn.Module):
    def __init__(self, channels=(8, 16)):
        super().__init__()
        self.conv1 = nn.Conv2d(1, channels[0], 3)
        self.conv2 = nn.Conv2d(channels[0], channels[1], 3)
        self.head = nn.Linear(channels[1] * 4 * 4, 3)

    def forward(self, x):
        return self.head(F.relu(self.conv2(F.relu(self.conv1(x)))).flatten(1))


# ---- data, seam, one code path
x, y = capstone_data.train_rows("shapes_supervised")
xv, yv = capstone_data.val_rows("shapes_supervised")
fitting = False
SEAM = set()


def seam_hook(module, args):
    if fitting:
        for row in args[0].detach().cpu():
            SEAM.add(capstone_data.split_of("shapes_supervised", row))


def fresh_run(config):
    torch.manual_seed(SEED)
    model = ShapeCNN(config["channels"]).to(torch.device(config["device"]))
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    model.register_forward_pre_hook(seam_hook)          # before any training
    return model, optimizer


def training_pool(config):
    if config["simulated_rows"] == 0:
        return x, y
    xs, ys = capstone_data.simulate("shapes_supervised", config["simulated_rows"], seed=1)
    return torch.cat([x, xs]), torch.cat([y, ys])


def train(config, model, optimizer, start_step, checkpoint_path):
    device = torch.device(config["device"])
    pool_x, pool_y = training_pool(config)
    losses = []
    for step in range(start_step + 1, config["steps"] + 1):
        idx = torch.randint(0, pool_x.shape[0], (config["batch_size"],))
        out = train_step(model, (pool_x[idx], pool_y[idx]), optimizer, device, config["amp_dtype"])
        losses.append(out["loss"])
        if step == config["checkpoint_at"]:
            save_checkpoint(checkpoint_path, model, optimizer, step)
    return losses


def accuracy(model, images, labels):
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        acc = (model(images.to(device)).argmax(dim=1).cpu() == labels).double().mean().item()
    model.train()
    return acc

In [ ]:
CONFIG = CPU_CONFIG                    # the accelerator extension replaces only this dict

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "ckpt150.pt")
    other_path = os.path.join(tmp, "resumed.pt")

    # Uninterrupted run
    model, optimizer = fresh_run(CONFIG)
    fitting = True
    losses = train(CONFIG, model, optimizer, 0, path)
    fitting = False

    # Resumed run in a "new session"
    model2, optimizer2 = fresh_run(CONFIG)
    torch.manual_seed(SEED + 99)
    k = load_checkpoint(path, model2, optimizer2)
    fitting = True
    resumed = train(CONFIG, model2, optimizer2, k, other_path)
    fitting = False

states_equal = all(torch.equal(a, b) for a, b in zip(model.state_dict().values(), model2.state_dict().values()))
first20, last20 = sum(losses[:20]) / 20, sum(losses[-20:]) / 20
val_acc = accuracy(model2, xv, yv)
print(f"k = {k}; {len(losses)} losses; first-20 mean {first20:.4f}, last-20 mean {last20:.4f}, ratio {last20 / first20:.4f}")
print("resumed == losses[150:]:", resumed == losses[150:], "| final state_dicts equal:", states_equal)
print(f"validation accuracy of the resumed model: {val_acc:.4f} | seam: {SEAM}")

**Locked test score (one call, after all fitting).** The resumed model is scored.

In [ ]:
model2.eval()


def predict_fn(images):
    device = next(model2.parameters()).device
    with torch.no_grad():
        return model2(images.to(device)).argmax(dim=1).cpu().to(torch.int64)


assert capstone_data.test_call_count("shapes_supervised") == 0
test = capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_supervised")
assert capstone_data.test_call_count("shapes_supervised") == 1
print(f"test accuracy {test.score:.4f}, 95% bootstrap CI [{test.ci_low:.4f}, {test.ci_high:.4f}] over {test.n_examples} examples")

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

The same `train(config, ...)` and fresh-run code run on a Colab L4 with `L4_CONFIG` in place of `CPU_CONFIG`; it has exactly the same keys, so only the dictionary changes. With `simulated_rows = 60000`, `train` samples `idx` over `x, y` plus 60,000 rows simulated from the same generative process (`capstone_data.simulate("shapes_supervised", 60000, seed=1)`). Simulated rows hash to `None`, never to `"val"` or `"test"`, so the seam check becomes "no `val` or `test` rows".

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "batch_size": 1024, "steps": 6000,
             "checkpoint_at": 3000, "lr": 0.003, "channels": (32, 64), "simulated_rows": 60000}
```

**What to observe.** Validation accuracy versus the number of simulated rows; steps per second with `amp_dtype=None` versus `torch.bfloat16`; peak memory against Session 1's estimate for `channels=(32, 64)`; and whether the resumed losses match the uninterrupted ones exactly on the GPU (they may not, without deterministic algorithms).

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
assert CPU_CONFIG["amp_dtype"] is None
assert k == 150
assert len(losses) == 300 and len(resumed) == 150 and all(type(v) is float for v in resumed)
assert resumed == losses[150:]
assert states_equal
assert last20 <= 0.25 * first20
assert val_acc >= 0.75
assert SEAM == {"train"}
assert capstone_data.test_call_count("shapes_supervised") == 1
assert test.score >= 0.75 and test.ci_low <= test.score <= test.ci_high